# CLIP4CAD-GFA v4 Training

**Slot Attention with Query Distillation**

## Key Fix from v2.4
v2.4 self-grounding still collapsed because `SelfQueryGenerator` produces queries in a
**different semantic space** than text features. Even with shared grounding, if Q_self ≠ T_feat,
the embeddings diverge.

## v4 Solution: Query-Level Supervision
- Replace `SelfQueryGenerator` with `SlotAttentionQueryGenerator` (Locatello et al., 2020)
- **Query Distillation**: L_query = cosine_distance(Q_self, T_feat.detach())
- **Attention Distillation**: L_attn = KL(A_self || G_guided)
- Q_self is FORCED to match T_feat → Same queries + shared grounding = same embeddings!

## Training Stages
- **Stage 1 (Epochs 1-15)**: Heavy query distillation (λ_query=1.5)
- **Stage 2 (Epochs 16-35)**: Balanced training with hard negatives

## Loss Weights
| Stage | λ_self | λ_query | λ_attn | λ_embed | λ_detail |
|-------|--------|---------|--------|---------|----------|
| 1 | 0.05 | **1.5** | 0.5 | 0.3 | 0.0 |
| 2 | 0.3 | **0.8** | 0.3 | 0.3 | 0.3 |

## Expected Results
- Query alignment: **>0.7** (key metric!)
- Self-cos: **0.80-0.90** (vs 0.08 in v2.4)
- Text→BRep R@1 (self): **55-65%** (vs 0.05% in v2.4)
- Gap (guided - self): **<10%**

In [ ]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Data Paths
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
BREP_FILE = Path("c:/Users/User/Desktop/brep_features.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")
OUTPUT_DIR = Path("../outputs/gfa_v4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"PC file: {PC_FILE} (exists: {PC_FILE.exists()})")
print(f"BRep file: {BREP_FILE} (exists: {BREP_FILE.exists()})")
print(f"Text file: {TEXT_FILE} (exists: {TEXT_FILE.exists()})")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Cell 3: Load Data
from clip4cad.data.gfa_dataset import GFAMappedDataset

print("Loading datasets...")
print("=" * 60)

# Train dataset - LOAD TO RAM for fast training
print("\n[1/2] Loading TRAIN dataset to RAM...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    brep_file=str(BREP_FILE),
    num_rotations=1,
    load_to_memory=True,
    use_live_text=False,
)
print(f"Train: {len(train_dataset):,} samples in RAM")

# Val dataset - ON DISK (saves RAM)
print("\n[2/2] Loading VAL dataset (on disk)...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    brep_file=str(BREP_FILE),
    num_rotations=1,
    load_to_memory=False,
    use_live_text=False,
)
print(f"Val: {len(val_dataset):,} samples on disk")

print("\n" + "=" * 60)
print("DATASETS READY!")

In [ ]:
# Cell 4: Verify Dataset
sample = train_dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"  brep_face_features: {sample['brep_face_features'].shape}")
print(f"  brep_edge_features: {sample['brep_edge_features'].shape}")
print(f"  pc_features: {sample['pc_features'].shape}")
print(f"  desc_embedding: {sample['desc_embedding'].shape}")

In [ ]:
# Cell 5: Create Model
from clip4cad.models import CLIP4CAD_GFA_v4, GFAv4Config
from clip4cad.losses import GFAv4Loss
from clip4cad.losses.gfa_v4_losses import compute_self_grounding_quality, compute_query_alignment

# Configuration
config = GFAv4Config(
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    d_unified=256,
    d_proj=128,
    d_ground=128,
    num_slots=12,
    num_detail_queries=8,
    num_heads=8,
    num_parser_layers=2,
    num_slot_iterations=3,    # Slot attention iterations
    slot_hidden_dim=512,      # Slot attention MLP dim
    dropout=0.1,
)

model = CLIP4CAD_GFA_v4(config).to(device)
print(f"Model: CLIP4CAD-GFA v4 (Slot Attention + Query Distillation)")
print(f"Parameters: {model.count_parameters():,}")
print(f"Trainable: {model.count_parameters(trainable_only=True):,}")
print(f"\nSlot Attention: {config.num_slot_iterations} iterations, hidden_dim={config.slot_hidden_dim}")

In [ ]:
# Cell 6: Training Configuration
from clip4cad.data.gfa_dataset import gfa_collate_fn

# Hyperparameters
BATCH_SIZE = 512
STAGE1_EPOCHS = 15
STAGE2_EPOCHS = 20
STAGE1_LR = 3e-5
STAGE2_LR = 1e-5
WARMUP_EPOCHS = 3
MAX_GRAD_NORM = 1.0

# v4 Loss weights - Query Distillation focused!
# Stage 1: Heavy query distillation (THE KEY FIX)
STAGE1_LAMBDA_SELF = 0.05             # Very low - don't let self compete
STAGE1_LAMBDA_QUERY = 1.5             # HIGH - THE KEY FIX!
STAGE1_LAMBDA_ATTN = 0.5              # Attention pattern distillation
STAGE1_LAMBDA_EMBED = 0.3             # Reduced (query distill handles alignment)
STAGE1_LAMBDA_DETAIL = 0.0

# Stage 2: Balanced
STAGE2_LAMBDA_SELF = 0.3
STAGE2_LAMBDA_QUERY = 0.8             # Still important
STAGE2_LAMBDA_ATTN = 0.3
STAGE2_LAMBDA_EMBED = 0.3
STAGE2_LAMBDA_DETAIL = 0.3

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
    collate_fn=gfa_collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=gfa_collate_fn,
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Total epochs: {STAGE1_EPOCHS + STAGE2_EPOCHS}")
print(f"\nStage 1 weights: self={STAGE1_LAMBDA_SELF}, query={STAGE1_LAMBDA_QUERY}, attn={STAGE1_LAMBDA_ATTN}, embed={STAGE1_LAMBDA_EMBED}")
print(f"Stage 2 weights: self={STAGE2_LAMBDA_SELF}, query={STAGE2_LAMBDA_QUERY}, attn={STAGE2_LAMBDA_ATTN}, embed={STAGE2_LAMBDA_EMBED}")

In [ ]:
# Cell 7: Initialize Optimizer, Loss, Scheduler
from clip4cad.training.hard_negative_mining import HardNegativeMiner

optimizer = AdamW(
    model.parameters(),
    lr=STAGE1_LR,
    weight_decay=0.01,
    betas=(0.9, 0.999),
)

criterion = GFAv4Loss(
    lambda_self=STAGE1_LAMBDA_SELF,
    lambda_query=STAGE1_LAMBDA_QUERY,
    lambda_attn=STAGE1_LAMBDA_ATTN,
    lambda_embed=STAGE1_LAMBDA_EMBED,
    lambda_detail=STAGE1_LAMBDA_DETAIL,
)

scaler = GradScaler()

# Learning rate scheduler with warmup
total_epochs = STAGE1_EPOCHS + STAGE2_EPOCHS
warmup_steps = WARMUP_EPOCHS * len(train_loader)
total_steps = total_epochs * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(1e-6 / STAGE1_LR, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)

# Hard negative miner (used in Stage 2)
hard_neg_miner = HardNegativeMiner(
    model=model,
    train_dataloader=train_loader,
    cache_dir=str(OUTPUT_DIR / "hard_negatives"),
    k=20,
    text_sim_threshold=0.8,
    min_negatives=1,
    max_negatives=10,
    use_faiss=True,
    device=str(device),
)
hard_negatives = None
MINE_EVERY_N_EPOCHS = 5

print("Optimizer, loss, scheduler, and hard negative miner initialized.")
print(f"\nKEY: Using Query Distillation (lambda_query={STAGE1_LAMBDA_QUERY}) to force Q_self → T_feat")

In [ ]:
# Cell 8: Training Loop

# Training state
global_step = 0
best_val_loss = float('inf')
best_self_cosine = 0.0
best_query_align = 0.0
current_stage = 1

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'self_cosine_brep': [],
    'self_cosine_pc': [],
    'query_loss': [],
    'attn_loss': [],
    'query_align_brep': [],
    'query_align_pc': [],
}

print("Starting training...")
print("="*70)

for epoch in range(1, total_epochs + 1):
    # Stage transition
    if epoch == STAGE1_EPOCHS + 1:
        print("\n" + "="*70)
        print("TRANSITIONING TO STAGE 2")
        print("="*70)
        current_stage = 2
        
        # Update loss weights
        criterion.update_weights(
            lambda_self=STAGE2_LAMBDA_SELF,
            lambda_query=STAGE2_LAMBDA_QUERY,
            lambda_attn=STAGE2_LAMBDA_ATTN,
            lambda_embed=STAGE2_LAMBDA_EMBED,
            lambda_detail=STAGE2_LAMBDA_DETAIL,
        )
        print(f"Updated loss weights: self={STAGE2_LAMBDA_SELF}, query={STAGE2_LAMBDA_QUERY}, attn={STAGE2_LAMBDA_ATTN}, detail={STAGE2_LAMBDA_DETAIL}")
        
        # Reduce learning rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = STAGE2_LR
        print(f"Reduced LR to {STAGE2_LR}")
        
        # Save Stage 1 checkpoint
        torch.save({
            'epoch': epoch - 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_self_cosine': best_self_cosine,
            'best_query_align': best_query_align,
        }, OUTPUT_DIR / 'checkpoint_stage1_final.pt')
        print(f"Saved Stage 1 checkpoint")
        
        # Mine hard negatives at start of Stage 2
        print("\nMining hard negatives for Stage 2...")
        hard_negatives = hard_neg_miner.mine(epoch=epoch)
        print(f"Mined hard negatives for {len(hard_negatives)} samples")
    
    # Re-mine hard negatives every N epochs in Stage 2
    if current_stage == 2 and epoch > STAGE1_EPOCHS + 1:
        if (epoch - STAGE1_EPOCHS - 1) % MINE_EVERY_N_EPOCHS == 0:
            print(f"\nRe-mining hard negatives (epoch {epoch})...")
            hard_negatives = hard_neg_miner.mine(epoch=epoch)
            print(f"Re-mined hard negatives for {len(hard_negatives)} samples")
    
    # Train epoch
    model.train()
    epoch_loss = 0.0
    epoch_self_cos_brep = []
    epoch_self_cos_pc = []
    epoch_query_loss = []
    epoch_attn_loss = []
    epoch_query_align_brep = []
    epoch_query_align_pc = []
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} (Stage {current_stage})")
    for batch_idx, batch in enumerate(pbar):
        # Get hard negatives for this batch (if in Stage 2)
        batch_hard_negs = None
        if current_stage == 2 and hard_negatives is not None:
            batch_size = batch['brep_face_features'].shape[0]
            start_idx = batch_idx * BATCH_SIZE
            batch_hard_negs = []
            for i in range(batch_size):
                sample_idx = start_idx + i
                if sample_idx in hard_negatives:
                    batch_hard_negs.append(hard_negatives[sample_idx])
                else:
                    batch_hard_negs.append(None)
        
        with autocast():
            outputs = model(batch)
            loss, loss_dict = criterion(outputs, hard_negatives=batch_hard_negs, stage=current_stage)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        global_step += 1
        epoch_loss += loss_dict['total']
        epoch_query_loss.append(loss_dict.get('query', 0))
        epoch_attn_loss.append(loss_dict.get('attn', 0))
        
        # Compute self-grounding quality
        if outputs.get('z_brep') is not None and outputs.get('z_brep_self') is not None:
            cos_brep = compute_self_grounding_quality(
                outputs['z_brep'].detach(),
                outputs['z_brep_self'].detach()
            )
            epoch_self_cos_brep.append(cos_brep)
        
        if outputs.get('z_pc') is not None and outputs.get('z_pc_self') is not None:
            cos_pc = compute_self_grounding_quality(
                outputs['z_pc'].detach(),
                outputs['z_pc_self'].detach()
            )
            epoch_self_cos_pc.append(cos_pc)
        
        # Compute query alignment (KEY METRIC for v4!)
        if outputs.get('T_feat') is not None and outputs.get('Q_brep_self') is not None:
            q_align_brep = compute_query_alignment(
                outputs['T_feat'].detach(),
                outputs['Q_brep_self'].detach(),
                outputs.get('confidence').detach() if outputs.get('confidence') is not None else None
            )
            epoch_query_align_brep.append(q_align_brep)
        
        if outputs.get('T_feat') is not None and outputs.get('Q_pc_self') is not None:
            q_align_pc = compute_query_alignment(
                outputs['T_feat'].detach(),
                outputs['Q_pc_self'].detach(),
                outputs.get('confidence').detach() if outputs.get('confidence') is not None else None
            )
            epoch_query_align_pc.append(q_align_pc)
        
        # Update progress bar
        postfix = {
            'loss': f"{loss_dict['total']:.3f}",
            'guided': f"{loss_dict['guided']:.3f}",
            'query': f"{loss_dict.get('query', 0):.3f}",
            'q_align': f"{epoch_query_align_brep[-1]:.3f}" if epoch_query_align_brep else "N/A",
            'cos': f"{epoch_self_cos_brep[-1]:.3f}" if epoch_self_cos_brep else "N/A",
        }
        if current_stage == 2:
            postfix['detail'] = f"{loss_dict.get('detail', 0):.3f}"
        pbar.set_postfix(postfix)
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_cos_brep = sum(epoch_self_cos_brep) / len(epoch_self_cos_brep) if epoch_self_cos_brep else 0
    avg_cos_pc = sum(epoch_self_cos_pc) / len(epoch_self_cos_pc) if epoch_self_cos_pc else 0
    avg_query_loss = sum(epoch_query_loss) / len(epoch_query_loss) if epoch_query_loss else 0
    avg_attn_loss = sum(epoch_attn_loss) / len(epoch_attn_loss) if epoch_attn_loss else 0
    avg_query_align_brep = sum(epoch_query_align_brep) / len(epoch_query_align_brep) if epoch_query_align_brep else 0
    avg_query_align_pc = sum(epoch_query_align_pc) / len(epoch_query_align_pc) if epoch_query_align_pc else 0
    
    history['train_loss'].append(avg_loss)
    history['self_cosine_brep'].append(avg_cos_brep)
    history['self_cosine_pc'].append(avg_cos_pc)
    history['query_loss'].append(avg_query_loss)
    history['attn_loss'].append(avg_attn_loss)
    history['query_align_brep'].append(avg_query_align_brep)
    history['query_align_pc'].append(avg_query_align_pc)
    
    if avg_cos_brep > best_self_cosine:
        best_self_cosine = avg_cos_brep
    if avg_query_align_brep > best_query_align:
        best_query_align = avg_query_align_brep
    
    print(f"\nEpoch {epoch}: Loss={avg_loss:.4f}, Self-cos={avg_cos_brep:.4f}, Query-align={avg_query_align_brep:.4f}, Query-loss={avg_query_loss:.4f}")
    print(f"Best: self-cos={best_self_cosine:.4f}, query-align={best_query_align:.4f}")
    
    # Validation every 5 epochs
    if epoch % 5 == 0:
        model.eval()
        val_loss = 0.0
        val_cos_brep = []
        val_query_align = []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                with autocast():
                    outputs = model(batch)
                    loss, loss_dict = criterion(outputs, stage=current_stage)
                val_loss += loss_dict['total']
                
                if outputs.get('z_brep') is not None and outputs.get('z_brep_self') is not None:
                    cos_brep = compute_self_grounding_quality(
                        outputs['z_brep'],
                        outputs['z_brep_self']
                    )
                    val_cos_brep.append(cos_brep)
                
                if outputs.get('T_feat') is not None and outputs.get('Q_brep_self') is not None:
                    q_align = compute_query_alignment(
                        outputs['T_feat'],
                        outputs['Q_brep_self'],
                        outputs.get('confidence')
                    )
                    val_query_align.append(q_align)
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_cos = sum(val_cos_brep) / len(val_cos_brep) if val_cos_brep else 0
        avg_val_q_align = sum(val_query_align) / len(val_query_align) if val_query_align else 0
        
        history['val_loss'].append(avg_val_loss)
        print(f"Validation: Loss={avg_val_loss:.4f}, Self-cos={avg_val_cos:.4f}, Query-align={avg_val_q_align:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_val_loss': best_val_loss,
                'best_self_cosine': best_self_cosine,
                'best_query_align': best_query_align,
            }, OUTPUT_DIR / 'checkpoint_best.pt')
            print("Saved best model!")
    
    # Save checkpoint every 5 epochs
    if epoch % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_self_cosine': best_self_cosine,
            'best_query_align': best_query_align,
        }, OUTPUT_DIR / f'checkpoint_epoch{epoch}.pt')
    
    # Clear cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

print("\n" + "="*70)
print("Training Complete!")
print(f"Best self-grounding cosine: {best_self_cosine:.4f}")
print(f"Best query alignment: {best_query_align:.4f}")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*70)

In [ ]:
# Cell 9: Save Final Model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config.__dict__,
    'best_self_cosine': best_self_cosine,
    'best_query_align': best_query_align,
    'history': history,
}, OUTPUT_DIR / 'clip4cad_gfa_v4_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'clip4cad_gfa_v4_final.pt'}")

In [ ]:
# Cell 10: Plot Training History
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Loss plot
axes[0, 0].plot(history['train_loss'], label='Train Loss')
if history['val_loss']:
    val_epochs = list(range(5, len(history['train_loss']) + 1, 5))[:len(history['val_loss'])]
    axes[0, 0].plot(val_epochs, history['val_loss'], 'o-', label='Val Loss')
axes[0, 0].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Self-grounding quality plot
axes[0, 1].plot(history['self_cosine_brep'], label='BRep Self-Cosine')
axes[0, 1].plot(history['self_cosine_pc'], label='PC Self-Cosine')
axes[0, 1].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[0, 1].axhline(y=0.85, color='g', linestyle=':', label='Target (0.85)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Cosine Similarity')
axes[0, 1].set_title('Self-Grounding Quality')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Query alignment plot (KEY METRIC!)
axes[0, 2].plot(history['query_align_brep'], label='BRep Query Align', linewidth=2)
axes[0, 2].plot(history['query_align_pc'], label='PC Query Align', linewidth=2)
axes[0, 2].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[0, 2].axhline(y=0.7, color='g', linestyle=':', label='Target (0.7)')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Cosine Similarity')
axes[0, 2].set_title('Query Alignment (KEY METRIC!)')
axes[0, 2].legend()
axes[0, 2].grid(True)

# Query loss plot
axes[1, 0].plot(history['query_loss'], label='Query Loss', color='purple')
axes[1, 0].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Query Distillation Loss')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Attention loss plot
axes[1, 1].plot(history['attn_loss'], label='Attention Loss', color='orange')
axes[1, 1].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('KL Divergence')
axes[1, 1].set_title('Attention Distillation Loss')
axes[1, 1].legend()
axes[1, 1].grid(True)

# Combined view: Query align vs Self-cos
axes[1, 2].plot(history['query_align_brep'], label='Query Alignment', color='blue', linewidth=2)
axes[1, 2].plot(history['self_cosine_brep'], label='Self-Cosine', color='green', linewidth=2)
axes[1, 2].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('Cosine Similarity')
axes[1, 2].set_title('Query Alignment drives Self-Cosine')
axes[1, 2].legend()
axes[1, 2].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history_v4.png', dpi=150)
plt.show()

print(f"Plot saved to {OUTPUT_DIR / 'training_history_v4.png'}")